# CSV to Excel Converter v2.1
Watches Downloads folder for `export_data*.csv` files and converts them to styled Excel tables.

In [1]:
import os, time, logging
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import subprocess

logging.disable(logging.WARNING)

In [2]:

DOWNLOADS = os.path.expanduser("~/Downloads")

TAGET_FOLDER = os.path.join(os.path.expanduser("~/Downloads"), "")  


COLUMNS_TO_REMOVE = [

]


EXPORTED_FILE_NAME_COLUMN = ""


TARGET_FILE = "export_data.csv"

In [3]:
def csv_to_excel(csv_path):
    try:
        df = pd.read_csv(csv_path)


        if EXPORTED_FILE_NAME_COLUMN not in df.columns:
            print("NO Column_Name Found")
            file_name = "export_data_MISSING_Column"
        else:
            file_name = str(df[EXPORTED_FILE_NAME_COLUMN].iloc[0]).strip()

        columns_found = [c for c in COLUMNS_TO_REMOVE if c in df.columns]
        if columns_found:
            df = df.drop(columns=columns_found)
            print(f"Removed columns {columns_found}")

        ts = datetime.fromtimestamp(os.path.getctime(csv_path))
        date_folder = f"{ts.year}-{ts.month:02}-{ts.day:02}"
        out_dir = os.path.join(TAGET_FOLDER, date_folder)
        os.makedirs(out_dir, exist_ok=True)


        out_path = os.path.join(out_dir, f"{file_name}.xlsx")
        counter = 1
        while os.path.exists(out_path):
            out_path = os.path.join(out_dir, f"{file_name}_{counter}.xlsx")
            counter += 1

        df.to_excel(out_path, index=False)

   
        wb = load_workbook(out_path)
        ws = wb.active
        max_col_letter = get_column_letter(ws.max_column)
        table_range = f"a1:{max_col_letter}{ws.max_row}"
        table = Table(displayName="DataTable", ref=table_range)
        table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
        ws.add_table(table)

        for col in ws.columns:
            max_len = max((len(str(c.value)) for c in col if c.value), default=0)
            ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 2

        wb.save(out_path)
        subprocess.Popen(
            ["start", "", out_path],
            shell=True,
            stdout=subprocess.DEVNULL
        )
        print(f"File saved to: {out_path}")

    except Exception as e:
        print(f"Error : {e}")

In [4]:
class CSVHandler(FileSystemEventHandler):
    def process(self, path):
        filename = os.path.basename(path).lower()
        if filename.startswith("export_data") and filename.endswith(".csv"):
            print(f"Detected new file in {path}")
            time.sleep(1)
            csv_to_excel(path)

    def on_created(self, e): self.process(e.src_path)
    def on_moved(self, e): self.process(e.dest_path)

In [5]:
print(f"folder listening' {TARGET_FILE} ' in: {DOWNLOADS}")
observer = Observer()
observer.schedule(CSVHandler(), DOWNLOADS, recursive=False)
observer.start()
try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    observer.stop()
observer.join()

folder listening' export_data.csv ' in: C:\Users\Admin/Downloads
Detected new file in C:\Users\Admin/Downloads\export_data.csv
Error : No columns to parse from file
Detected new file in C:\Users\Admin/Downloads\export_data.csv
NO Column_Name Found
File saved to: C:\Users\Admin/Downloads\2026-04-19\export_data_MISSING_Column.xlsx
